# Laboratorio 7 - Spark MLlib · Persona B
### Filtros, calidad de datos y segmentación con KMeans
CC3066 Data Science, UVG, Semestre II 2026.

Este borrador cubre:

- **Ejercicio 1 (segunda mitad):** validación de códigos contra el diccionario, construcción de `antiguedad`, filtros en orden fijo con conteo de exclusiones, unicidad de claves, respuestas conceptuales y Parquet preparado de 2025 y 2026.
- **Ejercicio 4 (parte técnica):** KMeans con K = 2, 3, 4 y 5, con y sin salario, y elección de K.

**Requisitos previos:** los Parquet de `staging/` que genera el notebook de Persona A y los diccionarios en `working_dir/raw/diccionarios/`.

**Genera:** `parquet/prep_2025/`, `parquet/prep_2026/` y `parquet/perfiles_2025/` (asignación de cluster para que Persona C construya los perfiles).

## 0. Configuración

Igual a la de Persona A, más las importaciones adicionales que necesita esta parte. En el notebook final estas celdas se fusionan con las de la sección 0 de A.

In [ ]:
import os
import gc
import re
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl
from IPython.display import display, Markdown

from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

SEED = 42
MAX_FILAS_GRAFICO = 5_000   # tope de filas que se transfieren a pandas para dibujar

BASE_DIR      = Path(os.environ.get("LAB7_BASE", "/opt/app/working_dir"))
RAW_DIR       = BASE_DIR / "raw"
DICC_DIR      = RAW_DIR / "diccionarios"
PARQUET_DIR   = BASE_DIR / "parquet"
STAGING_DIR   = PARQUET_DIR / "staging"
PREP_2025_DIR = PARQUET_DIR / "prep_2025"
PREP_2026_DIR = PARQUET_DIR / "prep_2026"
PERFILES_2025_DIR = PARQUET_DIR / "perfiles_2025"
MODELS_DIR    = BASE_DIR / "models"
FIG_DIR       = BASE_DIR / "figures"

for d in (RAW_DIR, STAGING_DIR, MODELS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid")

In [ ]:
spark = (
    SparkSession.builder
    .appName("lab7-spark-mllib")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    # Arrow desactivado: con algunas versiones de Java falla en toPandas(); los volúmenes son pequeños
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
assert spark.version.startswith("3.5"), "El laboratorio requiere Spark 3.5.x"

### 0.1 Catálogo de períodos y columnas

Mismo catálogo que en el notebook de A (el período sale del archivo de procedencia, no de `TRIMESTRE`). `registros_esperados` son los conteos publicados en el enunciado.

In [ ]:
ARCHIVOS = pd.DataFrame([
    # periodo_archivo, anio_archivo, trimestre_calendario, uso, registros_esperados
    ("2025T1", 2025, 1, "train",      51_588),
    ("2025T2", 2025, 2, "train",      51_167),
    ("2025T3", 2025, 3, "train",      51_583),
    ("2025T4", 2025, 4, "validacion", 49_338),
    ("2026T1", 2026, 1, "test",       49_843),
], columns=["periodo_archivo", "anio_archivo", "trimestre_calendario", "uso", "registros_esperados"])

PERIODOS      = ARCHIVOS["periodo_archivo"].tolist()
PERIODOS_2025 = ARCHIVOS.loc[ARCHIVOS["anio_archivo"] == 2025, "periodo_archivo"].tolist()

# Columnas analíticas que produce el staging de A (contrato entre secciones)
COLS_ANALITICAS = ["salario_mensual", "edad", "antiguedad_anios", "antiguedad_meses", "horas_semanales",
                   "nivel_educativo", "categoria_ocupacional", "dominio", "ocupado",
                   "NUM_HOGAR", "NUM_PERSONA", "FACTOR", "ANIO", "TRIMESTRE"]
COLS_META = ["periodo_archivo", "anio_archivo", "trimestre_calendario", "archivo_origen"]
ORDEN_SALIDA = COLS_ANALITICAS + ["antiguedad"] + COLS_META
CLAVE = ["periodo_archivo", "NUM_HOGAR", "NUM_PERSONA"]

## 1. Carga, armonización y calidad de datos (segunda mitad)

### 1.11 Códigos válidos según el diccionario

La validación de `nivel_educativo` (`P03A03A`), `categoria_ocupacional` (`P05C16`) y `dominio` (`DOMINIO`) se hace contra los diccionarios oficiales de cada archivo, que se leen aquí mismo desde `working_dir/raw/diccionarios/`, en lugar de escribir los códigos a mano. El período de cada diccionario se toma del nombre del archivo (por ejemplo `...ENEIC-IV-2025.xlsx` corresponde a `2025T4`).

En los diccionarios, la lista de valores aparece después de la fila *Valores de variable*: el nombre de la variable está solo en la primera fila de su bloque y las siguientes filas traen únicamente código y etiqueta. En IV de 2025 los códigos vienen como texto y en los demás como número, así que se normalizan igual que en el staging (`1`, `1.0` y `"01"` pasan a `"1"`).

In [ ]:
VARIABLES_CAT = {"P03A03A": "nivel_educativo", "P05C16": "categoria_ocupacional", "DOMINIO": "dominio"}

def normalizar_codigo(c) -> str:
    # 1, 1.0, "01", " 1 " -> "1"; texto no numérico se conserva en mayúsculas
    s = str(c).strip()
    try:
        d = float(s)
        if d == int(d):
            return str(int(d))
    except (ValueError, OverflowError):
        pass
    return s.upper()

def leer_diccionario(ruta: Path) -> dict:
    # Devuelve {variable original: {código normalizado: etiqueta}} solo para las variables categóricas usadas.
    wb = openpyxl.load_workbook(ruta, read_only=True, data_only=True)
    filas = list(wb.worksheets[0].iter_rows(values_only=True))
    wb.close()
    ini = next(i for i, r in enumerate(filas) if r[0] and str(r[0]).strip().lower().startswith("valores de variable"))
    valores, actual = {}, None
    for r in filas[ini + 2:]:                       # se salta el encabezado "Valor | Etiqueta"
        if r[0] is not None:
            actual = str(r[0]).strip().upper()
        if actual in VARIABLES_CAT and r[1] is not None:
            valores.setdefault(actual, {})[normalizar_codigo(r[1])] = str(r[2]).strip()
    return valores

def periodo_de_diccionario(nombre: str):
    m = re.search(r"ENEIC[-_ ]*(IV|III|II|I)[-_ ]*(20\d{2})", nombre, flags=re.I)
    if not m:
        return None
    trimestre = {"I": 1, "II": 2, "III": 3, "IV": 4}[m.group(1).upper()]
    return f"{m.group(2)}T{trimestre}"

dicc_por_periodo = {}
for ruta in sorted(DICC_DIR.glob("*.xlsx")):
    p = periodo_de_diccionario(ruta.name)
    if p in PERIODOS:
        dicc_por_periodo[p] = leer_diccionario(ruta)

faltan = [p for p in PERIODOS if p not in dicc_por_periodo]
assert not faltan, f"No se encontró diccionario para {faltan} en {DICC_DIR}"

# Los cinco diccionarios deben coincidir en las tres variables; si no, hay que decidir cuál aplica.
ref = dicc_por_periodo[PERIODOS[0]]
for p, d in dicc_por_periodo.items():
    assert d == ref, f"El diccionario de {p} difiere del de {PERIODOS[0]} en las variables categóricas"

CODIGOS = {VARIABLES_CAT[v]: ref[v] for v in VARIABLES_CAT}        # {'nivel_educativo': {'0': 'NINGUNO', ...}, ...}
CODIGOS_ASALARIADO = ["1", "2", "3", "4"]                           # definición del enunciado (P05C16)

tabla_codigos = pd.DataFrame(
    [(col, cod, etq) for col, d in CODIGOS.items() for cod, etq in d.items()],
    columns=["variable analítica", "código", "etiqueta en el diccionario"],
)
print(f"Diccionarios leídos: {sorted(dicc_por_periodo)} (coinciden en las tres variables)")
display(tabla_codigos)

Los cinco diccionarios coinciden, así que se usa un único conjunto de códigos válidos. Dos puntos a tener en cuenta:

- En `nivel_educativo` el código **`0` es "NINGUNO"**, una respuesta válida. Por eso no se trata como faltante ni se mezcla con `DESCONOCIDO`.
- `categoria_ocupacional` tiene nueve códigos válidos (1 a 9). Solo 1, 2, 3 y 4 son asalariados; 5 a 9 son códigos legítimos que se excluyen en el filtro de población, no en la validación.

### 1.12 Carga del staging y unicidad de claves

Se leen los Parquet de `staging/` de A y se unen los cuatro de 2025 con `unionByName`. El de 2026 queda separado. Antes de filtrar se verifica la unicidad de la combinación (`periodo_archivo`, `NUM_HOGAR`, `NUM_PERSONA`) sobre **todas** las filas de cada archivo.

Si una clave aparece más de una vez se clasifica según lo que difiera entre las filas repetidas, comparando las 14 columnas analíticas seleccionadas (el staging no conserva las 270 columnas originales):

- **Repetición exacta:** todas las copias son idénticas en esas 14 columnas.
- **Registros en conflicto:** la misma clave aparece con valores distintos.

No se usa `dropDuplicates()` para esconder el problema: primero se mide y clasifica, y la regla de qué hacer con los duplicados se aplica de forma explícita y contabilizada más adelante (1.16).

In [ ]:
faltan_staging = [p for p in PERIODOS if not (STAGING_DIR / p).exists()]
assert not faltan_staging, f"Falta el staging de {faltan_staging} en {STAGING_DIR}. Primero hay que correr el notebook de A."

staging = {p: spark.read.parquet(str(STAGING_DIR / p)) for p in PERIODOS}
for p, sdf in staging.items():
    faltan_cols = set(COLS_ANALITICAS + COLS_META) - set(sdf.columns)
    assert not faltan_cols, f"El staging de {p} no tiene {sorted(faltan_cols)}"

df_2025_raw = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=False),
                     [staging[p].select(*COLS_ANALITICAS, *COLS_META) for p in PERIODOS_2025])
df_2026_raw = staging["2026T1"].select(*COLS_ANALITICAS, *COLS_META)
df_todo_raw = df_2025_raw.unionByName(df_2026_raw).persist()

conteo = df_todo_raw.groupBy("periodo_archivo").count().toPandas().set_index("periodo_archivo")["count"]
verif = ARCHIVOS[["periodo_archivo", "uso", "registros_esperados"]].copy()
verif["filas_staging"] = verif["periodo_archivo"].map(conteo)
verif["coincide_con_enunciado"] = verif["filas_staging"] == verif["registros_esperados"]
display(verif)

In [ ]:
def resumen_por_clave(df: DataFrame) -> DataFrame:
    # Una fila por clave completa con cuántas filas trae y cuántas versiones distintas de la fila (14 columnas).
    firma = F.struct(*[F.col(c) for c in COLS_ANALITICAS])
    return (df.filter(F.col("NUM_HOGAR").isNotNull() & F.col("NUM_PERSONA").isNotNull())
              .groupBy(*CLAVE)
              .agg(F.count(F.lit(1)).alias("n_filas"),
                   F.size(F.collect_set(firma)).alias("n_versiones")))

def tabla_unicidad(df: DataFrame) -> pd.DataFrame:
    claves = resumen_por_clave(df)
    por_periodo = (claves.groupBy("periodo_archivo").agg(
        F.count(F.lit(1)).alias("claves_distintas"),
        F.sum((F.col("n_filas") > 1).cast("int")).alias("claves_repetidas"),
        F.sum(F.when(F.col("n_filas") > 1, F.col("n_filas") - 1).otherwise(0)).alias("filas_sobrantes"),
        F.sum(((F.col("n_filas") > 1) & (F.col("n_versiones") == 1)).cast("int")).alias("repeticiones_exactas"),
        F.sum(((F.col("n_filas") > 1) & (F.col("n_versiones") > 1)).cast("int")).alias("claves_en_conflicto"),
    ).toPandas().set_index("periodo_archivo"))
    filas = df.groupBy("periodo_archivo").agg(
        F.count(F.lit(1)).alias("filas"),
        F.sum((F.col("NUM_HOGAR").isNull() | F.col("NUM_PERSONA").isNull()).cast("int")).alias("filas_con_clave_nula"),
    ).toPandas().set_index("periodo_archivo")
    return filas.join(por_periodo).fillna(0).astype(int).sort_index()

unicidad = tabla_unicidad(df_todo_raw)
unicidad["clave_unica"] = (unicidad["claves_repetidas"] == 0) & (unicidad["filas_con_clave_nula"] == 0)
display(unicidad)

In [ ]:
def columnas_en_conflicto(df: DataFrame) -> pd.DataFrame:
    # Para las claves con versiones distintas, en cuántas claves difiere cada columna.
    conflictos = resumen_por_clave(df).filter(F.col("n_versiones") > 1).select(*CLAVE)
    sub = df.join(conflictos, CLAVE, "inner")
    por_clave = sub.groupBy(*CLAVE).agg(*[
        (F.size(F.collect_set(F.coalesce(F.col(c).cast("string"), F.lit("<nulo>")))) > 1).cast("int").alias(c)
        for c in COLS_ANALITICAS])
    res = por_clave.agg(*[F.sum(c).alias(c) for c in COLS_ANALITICAS]).toPandas().T
    res.columns = ["claves_donde_difiere"]
    return res[res["claves_donde_difiere"] > 0].sort_values("claves_donde_difiere", ascending=False)

n_conflictos = int(unicidad["claves_en_conflicto"].sum())
if n_conflictos:
    print(f"Hay {n_conflictos:,} claves en conflicto. Columnas en las que difieren las copias:")
    display(columnas_en_conflicto(df_todo_raw))
else:
    print("No hay claves en conflicto en ningún archivo.")

In [ ]:
n_rep = int(unicidad["claves_repetidas"].sum()); n_nulas = int(unicidad["filas_con_clave_nula"].sum())
n_exactas = int(unicidad["repeticiones_exactas"].sum())
if n_rep == 0 and n_nulas == 0:
    texto = ("**Lectura.** La combinación (`periodo_archivo`, `NUM_HOGAR`, `NUM_PERSONA`) es única en los cinco archivos: "
             "ninguna clave se repite y no hay claves nulas. No hay duplicados que clasificar, y el paso de "
             "deduplicación de la sección 1.16 no debería excluir registros.")
else:
    texto = (f"**Lectura.** Se encontraron {n_rep:,} claves repetidas ({n_exactas:,} repeticiones exactas y "
             f"{n_conflictos:,} claves en conflicto) y {n_nulas:,} filas con clave nula. Las repeticiones exactas no "
             "agregan información y no cambian ningún resultado. Los conflictos sí importan, porque la misma clave "
             "trae valores distintos (la tabla anterior indica en qué columnas). La regla acordada por el equipo se "
             "aplica y se contabiliza en la sección 1.16.")
display(Markdown(texto))

### 1.13 Validación de las variables categóricas

Cada código se compara con los válidos del diccionario. Los ausentes (nulos) y los no reconocidos se representan como **`DESCONOCIDO`**; no se convierten a cero ni a otro código. La tabla muestra qué llega en cada variable **antes de aplicar filtros**, sobre todos los registros de 2025.

In [ ]:
def mapear_categoricas(df: DataFrame) -> DataFrame:
    for col, validos in CODIGOS.items():
        df = df.withColumn(col, F.when(F.col(col).isin(*sorted(validos)), F.col(col)).otherwise(F.lit("DESCONOCIDO")))
    return df

def distribucion(df: DataFrame, col: str, etiquetas: dict) -> pd.DataFrame:
    d = (df.groupBy(F.coalesce(F.col(col), F.lit("<nulo>")).alias("codigo")).count().toPandas()
           .rename(columns={"count": "registros"}))
    d["etiqueta"] = d["codigo"].map(etiquetas).fillna("(no está en el diccionario)")
    d["variable"] = col
    d["reconocido"] = d["codigo"].isin(etiquetas.keys())
    return d.sort_values("codigo")[["variable", "codigo", "etiqueta", "reconocido", "registros"]]

antes = pd.concat([distribucion(df_2025_raw, col, etq) for col, etq in CODIGOS.items()], ignore_index=True)
antes["% de 2025"] = (100 * antes["registros"] / df_2025_raw.count()).round(2)
display(antes)

n_desc = int(antes.loc[~antes["reconocido"], "registros"].sum())
display(Markdown(f"**Lectura.** Antes de filtrar hay {n_desc:,} valores ausentes o no reconocidos entre las tres "
                 "variables (suma sobre todas las filas de 2025). Buena parte son saltos de flujo: por ejemplo, "
                 "`categoria_ocupacional` solo se pregunta a personas ocupadas. Todos pasan a `DESCONOCIDO`. Los de "
                 "`categoria_ocupacional` quedan fuera de la población en el paso 3 del filtrado; los de "
                 "`nivel_educativo` y `dominio` que permanezcan en la población analítica se conservan como una "
                 "categoría más y se cuantifican en la sección 1.16."))

### 1.14 Construcción de `antiguedad` y filtros en orden fijo

`antiguedad = antiguedad_anios + antiguedad_meses / 12`, en años. Se calcula para todas las filas; si falta alguno de los dos componentes, el resultado es nulo y el registro no permite evaluar los criterios de antigüedad.

Los filtros se aplican **siempre en el mismo orden**, tanto en 2025 como en 2026. Primero los que definen la población (pasos 1 a 4) y después los de calidad de las variables numéricas (pasos 5 a 8). Así, cada exclusión de calidad se cuenta solo entre registros que ya pertenecen a la población de análisis, y el paso 9 (claves repetidas) se evalúa entre los registros que superaron todo lo anterior.

| Paso | Criterio |
|---|---|
| 1 | Edad finita y mayor o igual a 15 |
| 2 | `OCUPADOS = 1` |
| 3 | `P05C16` en {1, 2, 3, 4} (asalariado) |
| 4 | Salario numérico, finito y estrictamente positivo |
| 5 | Componente de meses entero entre 0 y 11 |
| 6 | Antigüedad calculada mayor o igual a 0 |
| 7 | Antigüedad calculada menor o igual a la edad |
| 8 | Horas habituales mayores que 0 y menores o iguales a 168 |
| 9 | Clave (`periodo_archivo`, `NUM_HOGAR`, `NUM_PERSONA`) repetida: se conserva el primer registro |

Cada registro se marca con **el primer paso que incumple**, y se distingue por qué:

- **`incumple`:** el valor existe pero no cumple el criterio.
- **`no evaluable`:** falta el dato o no es un número finito (nulo, `NaN`, infinito), así que no se puede evaluar el criterio. Estos registros se **excluyen y se contabilizan**; el salario no se imputa.

Como cada fila cuenta en un solo paso, los conteos por paso suman exactamente los registros excluidos.

In [ ]:
INF = float("inf")

def es_finito(c: str):
    x = F.col(c)
    return x.isNotNull() & ~F.isnan(x) & (F.abs(x) != F.lit(INF))

# (paso, descripción, ¿se puede evaluar?, ¿cumple?)
PASOS = [
    (1, "Edad finita y >= 15",
        es_finito("edad"),
        F.col("edad") >= 15),
    (2, "OCUPADOS = 1",
        F.col("ocupado").isNotNull(),
        F.col("ocupado") == 1),
    (3, "P05C16 en {1,2,3,4} (asalariado)",
        F.col("categoria_ocupacional").isNotNull() & (F.col("categoria_ocupacional") != "DESCONOCIDO"),
        F.col("categoria_ocupacional").isin(*CODIGOS_ASALARIADO)),
    (4, "Salario numérico, finito y > 0",
        es_finito("salario_mensual"),
        F.col("salario_mensual") > 0),
    (5, "Meses enteros entre 0 y 11",
        es_finito("antiguedad_meses"),
        (F.col("antiguedad_meses") == F.floor("antiguedad_meses")) & F.col("antiguedad_meses").between(0, 11)),
    (6, "Antigüedad >= 0",
        es_finito("antiguedad"),
        F.col("antiguedad") >= 0),
    (7, "Antigüedad <= edad",
        es_finito("antiguedad") & es_finito("edad"),
        F.col("antiguedad") <= F.col("edad")),
    (8, "0 < horas semanales <= 168",
        es_finito("horas_semanales"),
        (F.col("horas_semanales") > 0) & (F.col("horas_semanales") <= 168)),
]
DESC_PASOS = {n: desc for n, desc, _, _ in PASOS}
DESC_PASOS[9] = "Clave repetida (se conserva el primer registro)"

def marcar_exclusiones(df: DataFrame) -> DataFrame:
    # Agrega antiguedad, paso_exclusion (primer paso que incumple; nulo si el registro se conserva) y motivo_exclusion.
    d = df.withColumn("antiguedad", F.col("antiguedad_anios") + F.col("antiguedad_meses") / 12)
    paso = motivo = None
    for n, _, evaluable, cumple in PASOS:
        ev = F.coalesce(evaluable, F.lit(False))
        ok = F.coalesce(cumple, F.lit(False))
        falla = ~ev | ~ok
        m = F.when(~ev, F.lit("no evaluable")).otherwise(F.lit("incumple"))
        paso   = F.when(falla, F.lit(n)) if paso   is None else paso.when(falla, F.lit(n))
        motivo = F.when(falla, m)        if motivo is None else motivo.when(falla, m)
    d = d.withColumn("paso_exclusion", paso).withColumn("motivo_exclusion", motivo)

    # Paso 9: entre los registros que superaron 1-8, se conserva el primero de cada clave.
    # "Primero" = primero según el orden ascendente de las 14 columnas analíticas (nulos primero), que es
    # determinista; el staging no guarda el número de fila del Excel. Con repeticiones exactas es indiferente.
    d = d.withColumn("_elegible", F.col("paso_exclusion").isNull())
    clave_completa = F.col("NUM_HOGAR").isNotNull() & F.col("NUM_PERSONA").isNotNull()
    w = Window.partitionBy(*CLAVE, "_elegible").orderBy(*[F.col(c).asc_nulls_first() for c in COLS_ANALITICAS])
    d = d.withColumn("_rn", F.row_number().over(w))
    repetido = F.col("_elegible") & clave_completa & (F.col("_rn") > 1)
    return (d.withColumn("paso_exclusion", F.when(repetido, F.lit(9)).otherwise(F.col("paso_exclusion")))
             .withColumn("motivo_exclusion", F.when(repetido, F.lit("clave repetida")).otherwise(F.col("motivo_exclusion")))
             .drop("_elegible", "_rn"))

# Mismas reglas y mismo orden para 2025 y 2026; el mapeo a DESCONOCIDO va antes de los filtros
marcado = marcar_exclusiones(mapear_categoricas(df_todo_raw)).persist()
print("Registros marcados:", f"{marcado.count():,}")

### 1.15 Registros por archivo antes y después de los filtros, y excluidos por paso

In [ ]:
def tabla_embudo(marcado: DataFrame, periodos: list) -> pd.DataFrame:
    cnt = marcado.groupBy("periodo_archivo", "paso_exclusion", "motivo_exclusion").count().toPandas()
    cnt["paso_exclusion"] = pd.to_numeric(cnt["paso_exclusion"])

    def serie(mask):
        s = cnt[mask].groupby("periodo_archivo")["count"].sum()
        return s.reindex(periodos).fillna(0).astype(int)

    filas = []
    def agregar(paso, desc, motivo, s):
        filas.append({"paso": paso, "criterio": desc, "motivo": motivo, **s.to_dict(), "total": int(s.sum())})

    agregar(0, "Registros originales", "", serie(cnt["count"].notna()))
    for n, desc in DESC_PASOS.items():
        motivos = sorted(cnt.loc[cnt["paso_exclusion"] == n, "motivo_exclusion"].dropna().unique())
        if not motivos:
            agregar(n, desc, "-", serie(cnt["count"].isna()))          # paso sin exclusiones: fila de ceros
        for m in motivos:
            agregar(n, desc, m, serie((cnt["paso_exclusion"] == n) & (cnt["motivo_exclusion"] == m)))
    agregar(99, "Registros analíticos (se conservan)", "", serie(cnt["paso_exclusion"].isna()))
    return pd.DataFrame(filas)

embudo = tabla_embudo(marcado, PERIODOS)
print("Registros excluidos en cada paso (mismo orden en todos los archivos):")
display(embudo)

antes_despues = pd.DataFrame({
    "uso": ARCHIVOS.set_index("periodo_archivo")["uso"],
    "antes_de_filtros": embudo.loc[embudo["paso"] == 0, PERIODOS].iloc[0],
    "despues_de_filtros": embudo.loc[embudo["paso"] == 99, PERIODOS].iloc[0],
})
antes_despues["% que se conserva"] = (100 * antes_despues["despues_de_filtros"] / antes_despues["antes_de_filtros"]).round(1)
print("Registros por archivo antes y después de los filtros:")
display(antes_despues)

In [ ]:
# Cuántos registros quedan tras cada paso (acumulado)
excl_por_paso = embudo[(embudo["paso"] >= 1) & (embudo["paso"] <= 9)].groupby("paso")[PERIODOS + ["total"]].sum()
orig = embudo.loc[embudo["paso"] == 0, PERIODOS + ["total"]].iloc[0]
quedan = (orig - excl_por_paso.cumsum()).astype(int)
quedan.insert(0, "criterio", [DESC_PASOS[n] for n in quedan.index])
display(quedan)

tot_orig = int(orig["total"]); tot_fin = int(embudo.loc[embudo["paso"] == 99, "total"].iloc[0])
top = excl_por_paso["total"].idxmax()
calidad = int(excl_por_paso.loc[5:8, "total"].sum())
display(Markdown(
    f"**Lectura.** De {tot_orig:,} registros originales (2025 y 2026 juntos) quedan {tot_fin:,} "
    f"({100 * tot_fin / tot_orig:.1f}%). El paso que más excluye es el {top} ({DESC_PASOS[top]}) con "
    f"{int(excl_por_paso.loc[top, 'total']):,} registros. Es esperable que los pasos 1 a 3 concentren la exclusión, "
    "porque definen la población (personas de 15 años o más, ocupadas y asalariadas) y la mayoría de los "
    "registros de una encuesta de hogares no cumple esa definición. Los filtros de calidad de las variables numéricas "
    f"(pasos 5 a 8) excluyen en total {calidad:,} registros; el detalle por archivo está en la tabla de arriba. "
    f"El paso 4 excluye {int(excl_por_paso.loc[4, 'total']):,} registros por salario ausente o no positivo, sin imputar. "
    "Por eso los resultados se refieren solo a asalariados con salario positivo registrado."
))
display(Markdown(
    "**Nota sobre el paso 2.** Su motivo aparece como `no evaluable` porque, según el diccionario, `OCUPADOS` solo "
    "define el código 1 (población ocupada) y el campo queda vacío para todas las demás personas. Ese vacío se "
    "interpreta como *no pertenece a la población ocupada*, un salto de flujo, y no como un dato perdido."
))

### 1.16 Claves repetidas entre los registros elegibles y categorías `DESCONOCIDO`

**Regla acordada para claves repetidas (decisión de diseño del equipo):** si dentro de un período hay registros con la misma (`periodo_archivo`, `NUM_HOGAR`, `NUM_PERSONA`), se conserva el primero y se excluyen los demás, contabilizados como paso 9. Se aplica entre los registros que ya superaron los pasos 1 a 8, de modo que los conteos originales y los de los pasos anteriores no se alteran. La documentación de cuántos casos afecta y de qué tipo son (repetición exacta o conflicto) es la de la sección 1.12; aquí se mide su efecto sobre la población analítica.

In [ ]:
elegibles_antes_dedup = marcado.filter(F.col("paso_exclusion").isNull() | (F.col("paso_exclusion") == 9))
excluidos_9 = int(embudo.loc[embudo["paso"] == 9, "total"].sum())
print(f"Registros excluidos en el paso 9 (clave repetida): {excluidos_9:,}")

if excluidos_9 > 0:
    u = tabla_unicidad(elegibles_antes_dedup.drop("paso_exclusion", "motivo_exclusion"))
    print("Unicidad entre los registros que superaron los pasos 1-8:")
    display(u)
else:
    print("Entre los registros elegibles no hubo claves repetidas: el paso 9 no excluyó nada.")

# Categorías DESCONOCIDO dentro de la población analítica (ya mapeadas)
prep_todo = marcado.filter(F.col("paso_exclusion").isNull())
resumen_cat = []
for col in CODIGOS:
    d = (prep_todo.groupBy("anio_archivo", col).count().toPandas()
         .rename(columns={col: "codigo", "count": "registros"}))
    d.insert(0, "variable", col)
    resumen_cat.append(d)
resumen_cat = pd.concat(resumen_cat, ignore_index=True)
resumen_cat["etiqueta"] = [
    CODIGOS[v].get(c, "DESCONOCIDO") for v, c in zip(resumen_cat["variable"], resumen_cat["codigo"])]
display(resumen_cat.pivot_table(index=["variable", "codigo", "etiqueta"], columns="anio_archivo",
                                values="registros", aggfunc="sum", fill_value=0))

n_desc_pob = int(resumen_cat.loc[resumen_cat["codigo"] == "DESCONOCIDO", "registros"].sum())
display(Markdown(f"**Lectura.** En la población analítica hay {n_desc_pob:,} registros con alguna categoría `DESCONOCIDO` "
                 "(`nivel_educativo` o `dominio`). Si ese número es cero, la categoría `DESCONOCIDO` existe en el código "
                 "para el caso de que aparezca, pero no interviene en estos datos. El código educativo `0` "
                 "(NINGUNO) aparece como categoría propia y no como faltante."))

### 1.17 Parquet preparado de 2025 y de 2026

Se guardan por separado, con exactamente las mismas reglas. Además de las columnas analíticas se conservan `NUM_HOGAR`, `NUM_PERSONA`, `FACTOR`, `ANIO`, `TRIMESTRE` y las columnas de trazabilidad, y se agrega `antiguedad`. Las columnas auxiliares del filtrado (`paso_exclusion`, `motivo_exclusion`) no se guardan.

Después de escribir se **vuelve a leer** cada Parquet y se comprueba que todas las reglas se cumplen en el 100% de los registros y que la clave es única.

In [ ]:
def guardar_prep(df_marcado: DataFrame, anio: int, destino: Path) -> DataFrame:
    prep = (df_marcado.filter(F.col("paso_exclusion").isNull() & (F.col("anio_archivo") == anio))
                      .select(*ORDEN_SALIDA))
    prep.coalesce(1).write.mode("overwrite").parquet(str(destino))
    return spark.read.parquet(str(destino))

def verificar_prep(prep: DataFrame, anios: int) -> dict:
    # Cuenta violaciones de cada regla en el Parquet ya guardado. Todo debe ser 0.
    exprs = {f"paso_{n}": F.sum((~(F.coalesce(ev, F.lit(False)) & F.coalesce(ok, F.lit(False)))).cast("int"))
             for n, _, ev, ok in PASOS}
    exprs["anio_distinto"] = F.sum((F.col("anio_archivo") != anios).cast("int"))
    exprs["cat_no_valida"] = F.sum((
        ~F.col("nivel_educativo").isin(*CODIGOS["nivel_educativo"], "DESCONOCIDO")
        | ~F.col("dominio").isin(*CODIGOS["dominio"], "DESCONOCIDO")
        | ~F.col("categoria_ocupacional").isin(*CODIGOS_ASALARIADO)).cast("int"))
    viol = prep.agg(*[v.alias(k) for k, v in exprs.items()]).first().asDict()
    duplicadas = prep.groupBy(*CLAVE).count().filter("count > 1").count()
    viol["claves_repetidas"] = duplicadas
    return {k: int(v or 0) for k, v in viol.items()}

df_2025 = guardar_prep(marcado, 2025, PREP_2025_DIR)
df_2026 = guardar_prep(marcado, 2026, PREP_2026_DIR)

esperado = {2025: int(embudo.loc[embudo["paso"] == 99, PERIODOS_2025].iloc[0].sum()),
            2026: int(embudo.loc[embudo["paso"] == 99, "2026T1"].iloc[0])}
filas_guardadas = []
for anio, prep in ((2025, df_2025), (2026, df_2026)):
    v = verificar_prep(prep, anio)
    n = prep.count()
    assert n == esperado[anio], f"{anio}: {n} filas guardadas, se esperaban {esperado[anio]}"
    assert sum(v.values()) == 0, f"{anio}: hay violaciones de las reglas: {v}"
    filas_guardadas.append({"conjunto": f"prep_{anio}", "filas": n, "violaciones_de_reglas": sum(v.values()),
                            "columnas": len(prep.columns)})
display(pd.DataFrame(filas_guardadas))
df_2025.printSchema()
df_2025.show(5, truncate=False)

### 1.18 Respuestas conceptuales

**¿Por qué una persona observada en dos períodos no debe eliminarse como duplicado del conjunto longitudinal?**

La ENEIC tiene un diseño longitudinal con rotación: una parte de los hogares sale de la muestra y otra permanece, por lo que una misma persona puede aparecer en trimestres distintos. Cada aparición es una **observación distinta**, hecha en otro momento: la edad, el puesto, las horas y el salario pueden haber cambiado, y son justamente esos cambios lo que los datos registran. Un duplicado es una fila repetida dentro de un mismo período (la misma clave `periodo_archivo`, `NUM_HOGAR`, `NUM_PERSONA` más de una vez), que es lo que se verificó y trató en las secciones 1.12 y 1.16. Por eso la unidad de análisis es persona-período y el período forma parte de la clave. Eliminar a quien aparece en dos trimestres descartaría observaciones legítimas y sesgaría el conjunto hacia quienes aparecen una sola vez.

Esto tiene una consecuencia para el modelado: las observaciones de la misma persona no son independientes entre sí, y una persona puede estar tanto en el entrenamiento (2025) como en la prueba (2026). Es una limitación que conviene mencionar al interpretar el desempeño de los modelos.

**¿Por qué el número de registros de la base filtrada no representa a todos los trabajadores del país?**

1. **Es una muestra de hogares, no un censo.** Cada registro representa a un número distinto de personas de la población.
2. **La población filtrada es solo una parte de los ocupados:** asalariados con salario positivo registrado. Quedan fuera los trabajadores por cuenta propia, patronos y trabajadores no remunerados (`P05C16` 5 a 9), y los asalariados sin salario reportado o con datos inconsistentes.
3. **La no respuesta y los datos inválidos no son aleatorios.** Si quienes no reportan salario difieren en su salario de quienes sí lo hacen, la población con salario positivo registrado está sesgada.
4. **El análisis principal no está ponderado,** así que cada registro pesa lo mismo aunque represente a más o menos personas.

**¿Para qué se usaría `FACTOR` en un análisis poblacional?** Es el **factor de expansión**: indica, en términos generales, cuántas personas de la población representa cada registro (según su probabilidad de selección, con ajustes por no respuesta). En un análisis poblacional se usaría para estimar totales (la suma de `FACTOR` estima cuántas personas cumplen una condición), medias, proporciones y percentiles ponderados, y para corregir que dominios sobrerrepresentados en la muestra no pesen de más. Los errores estándar además requieren la información del diseño muestral (estratos y unidades primarias), que no está entre las variables seleccionadas. Aquí se conserva pero **no se usa** en clustering, modelos ni métricas, y por eso los resultados describen los registros analizados y no son estimaciones oficiales de la población guatemalteca.

In [ ]:
# Evidencia 1: registros de la muestra frente a población expandida (suma de FACTOR) por dominio, en 2025
pesos = (df_2025.groupBy("dominio")
         .agg(F.count(F.lit(1)).alias("registros"), F.sum("FACTOR").alias("suma_factor"),
              F.avg("FACTOR").alias("factor_medio"))
         .toPandas().sort_values("dominio"))
pesos["% de registros"] = (100 * pesos["registros"] / pesos["registros"].sum()).round(1)
pesos["% de suma FACTOR"] = (100 * pesos["suma_factor"] / pesos["suma_factor"].sum()).round(1)
pesos["etiqueta"] = pesos["dominio"].map(CODIGOS["dominio"])
display(pesos[["dominio", "etiqueta", "registros", "% de registros", "suma_factor", "% de suma FACTOR", "factor_medio"]].round(1))

pesos["diferencia"] = pesos["% de registros"] - pesos["% de suma FACTOR"]
sobre = pesos.loc[pesos["diferencia"].idxmax()]; sub = pesos.loc[pesos["diferencia"].idxmin()]
display(Markdown(
    f"**Lectura.** El dominio {sobre['etiqueta']} aporta {sobre['% de registros']:.1f}% de los registros pero solo "
    f"{sobre['% de suma FACTOR']:.1f}% de la población expandida (factor medio {sobre['factor_medio']:.0f}), mientras que "
    f"{sub['etiqueta']} aporta {sub['% de registros']:.1f}% de los registros y {sub['% de suma FACTOR']:.1f}% de la población "
    f"expandida (factor medio {sub['factor_medio']:.0f}). Contar registros sin ponderar sobrerrepresenta los dominios con "
    "factor de expansión bajo, y es una razón concreta para no presentar los conteos de esta base como cifras de población."))

In [ ]:
# Evidencia 2: claves (NUM_HOGAR, NUM_PERSONA) que aparecen en más de un período de 2025
a = df_2025.select("NUM_HOGAR", "NUM_PERSONA", F.col("periodo_archivo").alias("p_a"), F.col("edad").alias("edad_a"))
b = df_2025.select("NUM_HOGAR", "NUM_PERSONA", F.col("periodo_archivo").alias("p_b"), F.col("edad").alias("edad_b"))
pares = a.join(b, ["NUM_HOGAR", "NUM_PERSONA"]).filter(F.col("p_a") < F.col("p_b"))
n_pares = pares.count()
if n_pares == 0:
    print("No hay claves (NUM_HOGAR, NUM_PERSONA) en más de un período dentro de la población analítica de 2025.")
else:
    r = pares.agg(F.count(F.lit(1)).alias("n"),
                  F.avg((F.abs(F.col("edad_b") - F.col("edad_a")) <= 2).cast("int")).alias("p_edad_similar")).first()
    display(Markdown(f"**Lectura.** {r['n']:,} pares de registros de 2025 comparten (`NUM_HOGAR`, `NUM_PERSONA`) en períodos distintos "
                     f"y en el {100 * r['p_edad_similar']:.1f}% la edad difiere en 2 años o menos. Una coincidencia de números "
                     "por casualidad daría edades dispares, así que un porcentaje alto sugiere que son las mismas personas "
                     "observadas más de una vez. Es evidencia indirecta: los números de hogar y persona no se verifican "
                     "contra ningún identificador externo."))

---
## 4. Segmentación de perfiles con KMeans (selección de K)

Se agrupan los registros elegibles de 2025 (`prep_2025`, sin ponderar) para identificar perfiles de trabajadores asalariados. Esta sección cubre la parte técnica: variables, estandarización, K = 2, 3, 4 y 5, métricas y elección de K. La descripción de cada cluster (tamaños, medias, composición y nombres) la desarrolla Persona C sobre el K elegido aquí.

**Variables.** Se comparan dos variantes:

- **`sin_salario`:** `edad`, `antiguedad` y `horas_semanales`.
- **`con_salario`:** las anteriores más `salario_mensual`.

**Estandarización.** Las variables están en escalas muy distintas (años, horas y quetzales). KMeans usa distancia euclidiana, así que sin estandarizar dominaría la variable con mayor escala numérica (el salario, en miles). Se usa `StandardScaler` con media 0 y desviación 1, ajustado sobre los mismos registros que se agrupan.

**Métricas.** Por cada K y variante:

- **WSSSE** (costo de entrenamiento de KMeans, suma de distancias cuadradas de cada punto a su centro). Siempre baja al aumentar K; se busca el "codo" donde la mejora deja de ser grande.
- **Silueta** (`ClusteringEvaluator`, distancia euclidiana cuadrada), entre -1 y 1: qué tan cohesionados y separados están los clusters. Más alta es mejor.
- **Tamaño del cluster más pequeño**, para detectar clusters diminutos, típicamente formados por unos pocos valores extremos.

Ambas métricas se calculan sobre **todos** los registros, no sobre una muestra.

In [ ]:
VARIANTES = {
    "sin_salario": ["edad", "antiguedad", "horas_semanales"],
    "con_salario": ["salario_mensual", "edad", "antiguedad", "horas_semanales"],
}
K_LISTA = [2, 3, 4, 5]

df_km = spark.read.parquet(str(PREP_2025_DIR))
n_km = df_km.count()

evaluador = ClusteringEvaluator(featuresCol="features", predictionCol="cluster",
                                metricName="silhouette", distanceMeasure="squaredEuclidean")
resultados, modelos, escaladores = [], {}, {}

for variante, cols in VARIANTES.items():
    ensamblador = VectorAssembler(inputCols=cols, outputCol="features_raw", handleInvalid="error")
    base = ensamblador.transform(df_km.select(*cols))
    escaladores[variante] = StandardScaler(inputCol="features_raw", outputCol="features",
                                           withMean=True, withStd=True).fit(base)
    esc = escaladores[variante].transform(base).select("features").persist()
    for k in K_LISTA:
        modelo = KMeans(k=k, seed=SEED, featuresCol="features", predictionCol="cluster", maxIter=100).fit(esc)
        pred = modelo.transform(esc)
        tamanos = list(modelo.summary.clusterSizes)
        resultados.append({
            "variante": variante, "k": k,
            "wssse": modelo.summary.trainingCost,
            "silueta": evaluador.evaluate(pred),
            "iteraciones": modelo.summary.numIter,
            "tam_min": min(tamanos), "tam_max": max(tamanos),
            "prop_min": min(tamanos) / n_km,
            "tamanos": tamanos,
        })
        modelos[(variante, k)] = modelo
    esc.unpersist()

res_km = pd.DataFrame(resultados)
res_km["mejora_wssse_%"] = (res_km.groupby("variante")["wssse"].pct_change() * -100).round(1)
print(f"Registros agrupados (2025, todos los elegibles): {n_km:,}")
display(res_km.assign(prop_min=(100 * res_km["prop_min"]).round(1)).rename(columns={"prop_min": "cluster_menor_%"})
        .drop(columns="tamanos").round({"wssse": 1, "silueta": 4}))

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
colores = {"sin_salario": "#2b8cbe", "con_salario": "#e6550d"}
for i, variante in enumerate(VARIANTES):
    g = res_km[res_km["variante"] == variante]
    ejes[i, 0].plot(g["k"], g["wssse"], "o-", color=colores[variante])
    ejes[i, 0].set_ylabel("WSSSE (codo)"); ejes[i, 0].set_title(f"{variante}: WSSSE")
    ejes[i, 1].plot(g["k"], g["silueta"], "o-", color=colores[variante])
    ejes[i, 1].set_ylabel("Silueta"); ejes[i, 1].set_title(f"{variante}: silueta")
    for _, r in g.iterrows():
        ejes[i, 1].annotate(f"{r['silueta']:.3f}", (r["k"], r["silueta"]), textcoords="offset points",
                            xytext=(0, 6), ha="center", fontsize=8)
for ax in ejes[1]:
    ax.set_xlabel("K (número de clusters)"); ax.set_xticks(K_LISTA)
fig.suptitle(f"Selección de K - registros elegibles de 2025 (n = {n_km:,}), variables estandarizadas", y=1.0)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_seleccion_k.png", dpi=120, bbox_inches="tight")
plt.show()

### 4.1 Criterio de elección

El criterio del equipo, aplicado de forma explícita en el código:

1. **Variante principal: `sin_salario`.** Ver justificación en 4.2.
2. Entre los K probados de esa variante, se descartan los que dejan un cluster con menos del 5% de los registros (clusters tan pequeños suelen ser valores extremos, no perfiles).
3. Entre los restantes se elige el de **mayor silueta**. El WSSSE (codo) se usa como apoyo: no decide, porque siempre baja al aumentar K.

La silueta tiende a favorecer K pequeños, y un K = 2 puede separar solo "jóvenes" de "mayores" sin aportar perfiles útiles. Por eso la elección se muestra con todos los números y se puede sobrescribir manualmente con `K_MANUAL` si el equipo, al mirar la tabla, prefiere otro K por interpretabilidad.

In [ ]:
VARIANTE_ELEGIDA = "sin_salario"
MIN_PROP = 0.05
K_MANUAL = None            # poner un entero (2 a 5) para forzar otro K; None = aplicar el criterio

cand = res_km[(res_km["variante"] == VARIANTE_ELEGIDA) & (res_km["prop_min"] >= MIN_PROP)]
if cand.empty:
    cand = res_km[res_km["variante"] == VARIANTE_ELEGIDA]
    print("Ningún K cumple el mínimo de 5% por cluster; se usa la silueta máxima sin ese filtro.")
K_ELEGIDO = int(K_MANUAL) if K_MANUAL is not None else int(cand.sort_values("silueta", ascending=False).iloc[0]["k"])
assert K_ELEGIDO in K_LISTA

fila = res_km[(res_km["variante"] == VARIANTE_ELEGIDA) & (res_km["k"] == K_ELEGIDO)].iloc[0]
otros = res_km[(res_km["variante"] == VARIANTE_ELEGIDA) & (res_km["k"] != K_ELEGIDO)].sort_values("silueta", ascending=False)
mejora = res_km[(res_km["variante"] == VARIANTE_ELEGIDA)].set_index("k")["mejora_wssse_%"]

display(Markdown(
    f"**K elegido: {K_ELEGIDO}** (variante `{VARIANTE_ELEGIDA}`). Silueta {fila['silueta']:.3f}, WSSSE {fila['wssse']:,.0f}, "
    f"cluster más pequeño con {fila['tam_min']:,} registros ({100 * fila['prop_min']:.1f}% del total). "
    f"Tamaños de los clusters: {sorted(fila['tamanos'], reverse=True)}. "
    f"Los otros K de la misma variante tienen silueta de {', '.join(f'K={int(r.k)}: {r.silueta:.3f}' for r in otros.itertuples())}. "
    f"La mejora del WSSSE al pasar de K-1 a K es de {', '.join(f'K={int(k)}: {v:.1f}%' for k, v in mejora.dropna().items())}."
))
if K_ELEGIDO == 2 and K_MANUAL is None:
    display(Markdown("**Atención.** El criterio cayó en K = 2, que solo separa dos grupos amplios y puede ser poco útil como "
                     "*perfiles*. Conviene que el equipo mire la tabla y la gráfica: si el codo del WSSSE o una silueta similar "
                     "en un K mayor da grupos más interpretables, se puede fijar `K_MANUAL` y justificarlo en el Markdown."))

### 4.2 ¿Vale la pena incluir el salario en el clustering?

In [ ]:
cmp = (res_km.pivot(index="k", columns="variante", values="silueta").round(4)
       .join(res_km.pivot(index="k", columns="variante", values="prop_min").mul(100).round(1).add_prefix("cluster_menor_% "), rsuffix=""))
display(cmp)

sil_sin = res_km[res_km["variante"] == "sin_salario"]["silueta"].max()
sil_con = res_km[res_km["variante"] == "con_salario"]["silueta"].max()
min_con = res_km[res_km["variante"] == "con_salario"]["prop_min"].min() * 100
min_sin = res_km[res_km["variante"] == "sin_salario"]["prop_min"].min() * 100
display(Markdown(
    f"**Datos.** La mejor silueta es {sil_sin:.3f} sin salario y {sil_con:.3f} con salario. El cluster más pequeño llega a "
    f"{min_sin:.1f}% de los registros sin salario y a {min_con:.1f}% con salario. Las siluetas de las dos variantes no son "
    "estrictamente comparables, porque se calculan en espacios de distinta dimensión (3 y 4 variables)."
))

**Argumento.** Se propone **no incluir el salario** en la segmentación principal, por estas razones:

1. **El objetivo describe perfiles de trabajadores según edad, antigüedad y jornada.** El enunciado plantea la primera pregunta en esos términos. Incluir el salario haría que los clusters se definan en parte por lo que después se quiere comparar, y la diferencia de salario entre clusters sería circular en lugar de un hallazgo.
2. **Con salario, los extremos pesan mucho.** El salario está muy sesgado a la derecha: unos pocos salarios altos quedan a muchas desviaciones estándar de la media aun después de estandarizar, y KMeans tiende a dedicarles un cluster propio. El tamaño del cluster más pequeño de la tabla anterior mide justamente este riesgo.
3. **La etiqueta de cluster no puede usarse como predictor** en los modelos supervisados, y los clusters construidos con el salario incorporarían información del objetivo, lo que los haría inadecuados para cualquier uso posterior en la predicción.
4. **Sin salario, el salario sigue siendo útil como validación:** al calcular el salario mediano de cada cluster (lo hace Persona C) se puede ver si los perfiles se asocian con distinto nivel salarial sin haberlo usado para armarlos.

La variante `con_salario` se conserva como **análisis de sensibilidad**: sirve para mostrar en qué cambia la segmentación cuando el salario entra al agrupamiento.

Estos resultados describen asociaciones entre los registros analizados y no relaciones causales.

### 4.3 Asignación de cluster para Persona C

Se guarda `perfiles_2025/`: los registros de `prep_2025` con una columna `cluster` (0 a K-1) según la variante y el K elegidos. Es un archivo **aparte** de `prep_2025` a propósito: la etiqueta de cluster no debe entrar como predictor en los modelos supervisados, y así ningún pipeline la recoge por accidente.

In [ ]:
cols_km = VARIANTES[VARIANTE_ELEGIDA]
ens = VectorAssembler(inputCols=cols_km, outputCol="features_raw", handleInvalid="error").transform(df_km)
esc = escaladores[VARIANTE_ELEGIDA].transform(ens)
perfiles = (modelos[(VARIANTE_ELEGIDA, K_ELEGIDO)].transform(esc)
            .drop("features_raw", "features"))
perfiles.coalesce(1).write.mode("overwrite").parquet(str(PERFILES_2025_DIR))

chk = spark.read.parquet(str(PERFILES_2025_DIR))
assert chk.count() == n_km, "perfiles_2025 no tiene el mismo número de filas que prep_2025"
print(f"perfiles_2025 guardado: {chk.count():,} filas. Variante: {VARIANTE_ELEGIDA}, K = {K_ELEGIDO}")
display(chk.groupBy("cluster").count().orderBy("cluster").toPandas())

---
## Anexo. Prueba de las reglas con casos sintéticos

Los datos reales pueden no ejercitar todas las ramas del filtrado (por ejemplo, si ningún registro tiene meses fuera de rango). Esta celda construye registros artificiales, cada uno diseñado para incumplir **un** paso concreto, y comprueba con `assert` que la lógica de la sección 1.14 lo marca en el paso y con el motivo esperados, que el código educativo `0` no se trata como faltante y que la clasificación de duplicados distingue repeticiones exactas de conflictos.

In [ ]:
base = dict(salario_mensual=3000.0, edad=30.0, antiguedad_anios=2.0, antiguedad_meses=3.0, horas_semanales=44.0,
            nivel_educativo="4", categoria_ocupacional="2", dominio="1", ocupado=1, NUM_HOGAR=1, NUM_PERSONA=1,
            FACTOR=100.0, ANIO=2025, TRIMESTRE=3, periodo_archivo="TEST", anio_archivo=2025,
            trimestre_calendario=1, archivo_origen="sintetico.xlsx")

# (nombre, cambios, paso esperado, motivo esperado)
casos = [
    ("valido",               {},                                        None, None),
    ("edad 14",              dict(edad=14.0),                            1, "incumple"),
    ("edad nula",            dict(edad=None),                            1, "no evaluable"),
    ("edad NaN",             dict(edad=float("nan")),                    1, "no evaluable"),
    ("no ocupado",           dict(ocupado=None),                         2, "no evaluable"),
    ("no asalariado (5)",    dict(categoria_ocupacional="5"),            3, "incumple"),
    ("categoria nula",       dict(categoria_ocupacional=None),           3, "no evaluable"),
    ("categoria no valida",  dict(categoria_ocupacional="99"),           3, "no evaluable"),
    ("salario 0",            dict(salario_mensual=0.0),                  4, "incumple"),
    ("salario negativo",     dict(salario_mensual=-100.0),               4, "incumple"),
    ("salario nulo",         dict(salario_mensual=None),                 4, "no evaluable"),
    ("salario infinito",     dict(salario_mensual=float("inf")),         4, "no evaluable"),
    ("meses 12",             dict(antiguedad_meses=12.0),                5, "incumple"),
    ("meses 3.5",            dict(antiguedad_meses=3.5),                 5, "incumple"),
    ("meses nulo",           dict(antiguedad_meses=None),                5, "no evaluable"),
    ("anios negativo",       dict(antiguedad_anios=-1.0, antiguedad_meses=0.0), 6, "incumple"),
    ("anios nulo",           dict(antiguedad_anios=None),                6, "no evaluable"),
    ("antig > edad",         dict(edad=20.0, antiguedad_anios=25.0),     7, "incumple"),
    ("horas 0",              dict(horas_semanales=0.0),                  8, "incumple"),
    ("horas 169",            dict(horas_semanales=169.0),                8, "incumple"),
    ("horas nulas",          dict(horas_semanales=None),                 8, "no evaluable"),
    ("horas 168 (limite ok)", dict(horas_semanales=168.0),               None, None),
    ("edad 15 (limite ok)",  dict(edad=15.0, antiguedad_anios=0.0),      None, None),
    ("educacion 0 (ninguno)", dict(nivel_educativo="0"),                 None, None),
    ("educacion no valida",  dict(nivel_educativo="9"),                  None, None),
    ("educacion nula",       dict(nivel_educativo=None),                 None, None),
]
filas = []
for i, (nombre, cambios, _, _) in enumerate(casos, start=1):
    r = {**base, **cambios, "NUM_PERSONA": i}
    filas.append(r)
# Duplicados: par exacto (claves 900) y par en conflicto (claves 901)
filas += [{**base, "NUM_PERSONA": 900}, {**base, "NUM_PERSONA": 900},
          {**base, "NUM_PERSONA": 901, "salario_mensual": 2500.0}, {**base, "NUM_PERSONA": 901, "salario_mensual": 9000.0}]

esquema = df_todo_raw.schema
sint = spark.createDataFrame([tuple(r[f.name] for f in esquema.fields) for r in filas], schema=esquema)
res = marcar_exclusiones(mapear_categoricas(sint)).toPandas().set_index("NUM_PERSONA")

errores = []
for i, (nombre, cambios, paso_esp, motivo_esp) in enumerate(casos, start=1):
    paso = res.loc[i, "paso_exclusion"]; motivo = res.loc[i, "motivo_exclusion"]
    paso = None if pd.isna(paso) else int(paso); motivo = None if pd.isna(motivo) else motivo
    if (paso, motivo) != (paso_esp, motivo_esp):
        errores.append(f"{nombre}: esperado {(paso_esp, motivo_esp)}, obtenido {(paso, motivo)}")
assert not errores, "Reglas de filtrado con resultados inesperados:\n" + "\n".join(errores)

# Categorías: 0 se conserva; 9 y nulo pasan a DESCONOCIDO
idx = {c[0]: i for i, c in enumerate(casos, start=1)}
assert res.loc[idx["educacion 0 (ninguno)"], "nivel_educativo"] == "0"
assert res.loc[idx["educacion no valida"], "nivel_educativo"] == "DESCONOCIDO"
assert res.loc[idx["educacion nula"], "nivel_educativo"] == "DESCONOCIDO"

# Duplicados: exactamente uno de cada par se marca como paso 9, y el otro se conserva
for clave in (900, 901):
    par = res[res.index == clave]
    assert sorted(par["paso_exclusion"].fillna(0).astype(int).tolist()) == [0, 9], f"clave {clave}: {par['paso_exclusion'].tolist()}"

# Clasificación exacta vs conflicto
u = resumen_por_clave(sint.filter(F.col("NUM_PERSONA").isin(900, 901))).toPandas().set_index("NUM_PERSONA")
assert (u.loc[900, "n_filas"], u.loc[900, "n_versiones"]) == (2, 1), "el par 900 debe ser una repetición exacta"
assert (u.loc[901, "n_filas"], u.loc[901, "n_versiones"]) == (2, 2), "el par 901 debe ser un conflicto"

print(f"OK: {len(casos)} casos de filtrado, categorías y duplicados se comportan como se esperaba.")

In [ ]:
_ = df_todo_raw.unpersist()
_ = marcado.unpersist()